# Gemma 4 E4B IT — Dynamic Few-Shot Example Selection

Runs **dynamic_few_shot** and **dynamic_few_shot_cot** on the `facebook/natural_reasoning` sample: for each test question, the 3 demonstration examples are retrieved by embedding similarity (precomputed in `Data/dynamic_examples.json`) instead of using the fixed hand-written examples.

**Model**: `google/gemma-4-E4B-it` (4-bit quantized)

**Before running**, make sure Drive `NLP Project/Data/` contains `sampled.jsonl` and `dynamic_examples.json`.

Results are checkpointed to Drive after each technique, so if Colab disconnects you can rerun all cells and it resumes where it left off. Compare against the fixed-example runs with `notebook_comparison.ipynb`.

In [1]:
!pip install "transformers>=5.10.2" accelerate bitsandbytes
!pip install rouge-score tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=67393f259f51dad88de4b850c17c97184a5789df4c8a8e6b655652222d820d9f
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


In [2]:
import json
import os
import re
import time
import torch
import pandas as pd
from collections import Counter
from tqdm.notebook import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from rouge_score import rouge_scorer
from huggingface_hub import login
from google.colab import drive, userdata, files

In [3]:
login(token=userdata.get("HF_TOKEN"))
print("Logged into HuggingFace")

Logged into HuggingFace


## Configuration

In [4]:
MODEL_NAME = "google/gemma-4-E4B-it"
MODEL_TAG = "gemma"
DRIVE_DATA_PATH = "/content/drive/MyDrive/NLP Project/Data"
DRIVE_RESULTS_PATH = "/content/drive/MyDrive/NLP Project/Results"
SEED = 42
ANSWER_MARKER = "ANSWER:"

TECHNIQUES = ["dynamic_few_shot", "dynamic_few_shot_cot"]

MAX_TOKENS_BY_TYPE = {
    "single_word": 256,
    "short": 512,
    "long": 1024,
    "no_answer": 1024,
}

# Dynamic prompts are longer than the fixed ones (3 retrieved examples with
# reasoning), so the input budget is raised from 2048 to 3072.
MAX_INPUT_LENGTH = 3072

In [5]:
def get_batch_size(avg_input_tokens):
    if avg_input_tokens < 500:
        return 12
    elif avg_input_tokens < 1000:
        return 8
    elif avg_input_tokens < 1500:
        return 6
    elif avg_input_tokens < 2000:
        return 4
    elif avg_input_tokens < 2500:
        return 3
    else:
        return 2

## Dataset & Retrieved Examples

In [6]:
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
with open(f'{DRIVE_DATA_PATH}/sampled.jsonl', 'r', encoding='utf-8') as f:
    dataset = [json.loads(line) for line in f]

with open(f'{DRIVE_DATA_PATH}/dynamic_examples.json', 'r', encoding='utf-8') as f:
    dynamic_examples = {int(k): v for k, v in json.load(f).items()}

assert len(dataset) == 100, f"Expected 100 test questions, got {len(dataset)}"
missing = [item['sample_id'] for item in dataset if item['sample_id'] not in dynamic_examples]
assert not missing, f"Missing dynamic examples for sample_ids: {missing}"
bad = {sid: len(exs) for sid, exs in dynamic_examples.items() if len(exs) != 3}
assert not bad, f"Expected 3 examples per question, got: {bad}"

sims = [ex['similarity'] for exs in dynamic_examples.values() for ex in exs]
print(f"Loaded {len(dataset)} questions, 3 retrieved examples each")
print(f"Example similarity: mean={sum(sims)/len(sims):.3f}, min={min(sims):.3f}, max={max(sims):.3f}")

Loaded 100 questions, 3 retrieved examples each
Example similarity: mean=0.363, min=0.158, max=0.680


In [8]:
item = dataset[0]
print(f"TEST QUESTION (sample_id={item['sample_id']}, type={item['answer_type']}):")
print(item['question'][:400])
print()
for i, ex in enumerate(dynamic_examples[item['sample_id']], 1):
    print(f"--- Retrieved example {i} (pool_id={ex['pool_id']}, similarity={ex['similarity']:.3f}) ---")
    print('Q:', ex['question'][:200])
    print('A:', ex['answer'][:200])
    print()

TEST QUESTION (sample_id=0, type=short):
Given the information about Company B and Company D, including their contribution margins and fixed costs, calculate the breakeven point for each company and determine which company would reach its breakeven point first if both companies started selling their product at the same time.

--- Retrieved example 1 (pool_id=233, similarity=0.315) ---
Q: A hole is punched at a height h in the side of a container of height H, filled with water. Using Bernoulli's Equation and kinematic equations, derive an expression for the horizontal distance d that t
A: H/2

--- Retrieved example 2 (pool_id=344, similarity=0.329) ---
Q: Given two objects with movement equations \(x_1(t) = 2t + 1\) and \(y_1(t) = 4t^2\) for the first object, and \(x_2(t) = 3t\) and \(y_2(t) = 3t\) for the second object, determine whether these two obj
A: Solving the equations \(2t + 1 = 3t\) and \(4t^2 = 3t\) simultaneously.

--- Retrieved example 3 (pool_id=33, similarity=0.368) ---
Q

## Model

In [9]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

print(f"Model loaded: {MODEL_NAME}")
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"GPU memory reserved: {torch.cuda.memory_reserved() / 1e9:.2f} GB")

config.json:   0%|          | 0.00/5.14k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/17.3k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/16.0G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

Model loaded: google/gemma-4-E4B-it
GPU memory used: 9.52 GB
GPU memory reserved: 9.53 GB


## Prompting Techniques (dynamic examples)

In [10]:
def build_messages(question, technique, sample_id):
    examples = dynamic_examples[sample_id]

    if technique == "dynamic_few_shot":
        messages = [
            {"role": "system", "content": "You are a knowledgeable assistant. Answer questions thoroughly. State your final answer on a new line starting with 'ANSWER:'"}
        ]
        for ex in examples:
            messages.append({"role": "user", "content": ex["question"]})
            messages.append({"role": "assistant", "content": f"ANSWER: {ex['answer']}"})
        messages.append({"role": "user", "content": question})
        return messages

    elif technique == "dynamic_few_shot_cot":
        messages = [
            {"role": "system", "content": "You are a knowledgeable assistant. Think through problems step by step. State your final answer on a new line starting with 'ANSWER:'"}
        ]
        for ex in examples:
            messages.append({"role": "user", "content": ex["question"]})
            messages.append({"role": "assistant", "content": f"{ex['reasoning']}\n\nANSWER: {ex['answer']}"})
        messages.append({"role": "user", "content": question})
        return messages

In [11]:
print(f"Prompt token lengths (limit {MAX_INPUT_LENGTH}):")
for technique in TECHNIQUES:
    lengths = []
    for item in dataset:
        msgs = build_messages(item['question'], technique, item['sample_id'])
        templated = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        lengths.append(len(tokenizer.encode(templated)))
    over = sum(1 for l in lengths if l > MAX_INPUT_LENGTH)
    print(f"  {technique}: mean={sum(lengths)/len(lengths):.0f}, max={max(lengths)}, over limit={over}")
    assert over == 0, f"{over} prompts exceed MAX_INPUT_LENGTH and would be truncated!"
print("All prompts fit — nothing will be truncated.")

Prompt token lengths (limit 3072):
  dynamic_few_shot: mean=455, max=740, over limit=0
  dynamic_few_shot_cot: mean=1062, max=1470, over limit=0
All prompts fit — nothing will be truncated.


## Inference

In [12]:
def generate_batch(messages_list, max_new_tokens=1024):
    templated = [
        tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        for msgs in messages_list
    ]
    inputs = tokenizer(
        templated, return_tensors="pt", padding=True,
        truncation=True, max_length=MAX_INPUT_LENGTH
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.15,
            pad_token_id=tokenizer.eos_token_id,
        )

    responses = []
    for i, output in enumerate(outputs):
        prompt_len = inputs["input_ids"][i].shape[0]
        decoded = tokenizer.decode(output[prompt_len:], skip_special_tokens=True)
        responses.append(decoded.strip())
    return responses

In [13]:
os.makedirs(DRIVE_RESULTS_PATH, exist_ok=True)
CHECKPOINT_PATH = f"{DRIVE_RESULTS_PATH}/checkpoint_{MODEL_TAG}_dynamic.json"

results = []
if os.path.exists(CHECKPOINT_PATH):
    with open(CHECKPOINT_PATH, 'r', encoding='utf-8') as f:
        results = json.load(f)
    print(f"Loaded checkpoint with {len(results)} results")

done = {t for t in TECHNIQUES if sum(r['technique'] == t for r in results) == len(dataset)}
results = [r for r in results if r['technique'] in done]  # drop partial techniques
if done:
    print(f"Already complete (skipping): {sorted(done)}")

for technique in TECHNIQUES:
    if technique in done:
        continue
    torch.cuda.empty_cache()
    print(f"\n{'='*60}")
    print(f"Running: {technique}")
    print(f"GPU memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"{'='*60}")

    all_messages = [
        build_messages(item["question"], technique, item["sample_id"])
        for item in dataset
    ]
    templated_for_len = [
        tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        for msgs in all_messages
    ]
    input_lengths = [len(tokenizer.encode(t)) for t in templated_for_len]

    sorted_indices = sorted(range(len(dataset)), key=lambda i: input_lengths[i])
    all_messages = [all_messages[i] for i in sorted_indices]
    sorted_items = [dataset[i] for i in sorted_indices]
    sorted_lengths = [input_lengths[i] for i in sorted_indices]

    avg_len = sum(sorted_lengths) / len(sorted_lengths)
    batch_size = get_batch_size(avg_len)
    print(f"Avg input tokens: {avg_len:.0f} → batch size: {batch_size}")

    for batch_start in tqdm(range(0, len(dataset), batch_size), desc=technique):
        batch_end = min(batch_start + batch_size, len(dataset))
        batch_messages = all_messages[batch_start:batch_end]
        batch_items = sorted_items[batch_start:batch_end]

        batch_max_tokens = max(
            MAX_TOKENS_BY_TYPE.get(item["answer_type"], 1024)
            for item in batch_items
        )

        start = time.time()
        responses = generate_batch(batch_messages, max_new_tokens=batch_max_tokens)
        gen_time = (time.time() - start) / len(responses)

        for item, response in zip(batch_items, responses):
            examples = dynamic_examples[item["sample_id"]]
            results.append({
                "sample_id": item["sample_id"],
                "question": item["question"],
                "reference_answer": item["reference_answer"],
                "answer_type": item["answer_type"],
                "technique": technique,
                "response": response,
                "generation_time": gen_time,
                "response_length": len(response.split()),
                "example_ids": [ex["pool_id"] for ex in examples],
                "avg_example_similarity": round(sum(ex["similarity"] for ex in examples) / len(examples), 4),
            })

    with open(CHECKPOINT_PATH, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False)
    print(f"Checkpoint saved to Drive ({len(results)} results)")

df = pd.DataFrame(results)
print(f"\nDone: {len(df)} results ({len(dataset)} questions x {len(TECHNIQUES)} techniques)")


Running: dynamic_few_shot
GPU memory: 9.52 GB
Avg input tokens: 455 → batch size: 12


dynamic_few_shot:   0%|          | 0/9 [00:00<?, ?it/s]

Checkpoint saved to Drive (100 results)

Running: dynamic_few_shot_cot
GPU memory: 9.53 GB
Avg input tokens: 1062 → batch size: 6


dynamic_few_shot_cot:   0%|          | 0/17 [00:00<?, ?it/s]

Checkpoint saved to Drive (200 results)

Done: 200 results (100 questions x 2 techniques)


In [14]:
out_path = f"/content/results_{MODEL_TAG}_dynamic.json"
df.to_json(out_path, orient="records", indent=2)
df.to_json(f"{DRIVE_RESULTS_PATH}/results_{MODEL_TAG}_dynamic.json", orient="records", indent=2)
print(f"Saved to Drive: {DRIVE_RESULTS_PATH}/results_{MODEL_TAG}_dynamic.json")
files.download(out_path)

Saved to Drive: /content/drive/MyDrive/NLP Project/Results/results_gemma_dynamic.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
del model
torch.cuda.empty_cache()
print(f"GPU memory after cleanup: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

GPU memory after cleanup: 2.52 GB


## Quick Sanity Metrics

In [16]:
def normalize_latex(text):
    text = re.sub(r'\\frac\{([^}]+)\}\{([^}]+)\}', r'\1/\2', text)
    text = re.sub(r'\\boxed\{([^}]+)\}', r'\1', text)
    text = re.sub(r'\$+', '', text)
    text = re.sub(r'\\(left|right|quad|qquad|text|mathrm|mathbf|displaystyle)\b', '', text)
    text = re.sub(r'\\{2,}', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def extract_answer(response):
    if ANSWER_MARKER in response:
        return response.split(ANSWER_MARKER)[-1].strip()

    boxed = re.findall(r'\\boxed\{([^}]+)\}', response)
    if boxed:
        return boxed[-1]

    patterns = [
        r'(?:the\s+)?(?:final\s+)?answer\s+is[:\s]+(.+?)(?:\.|$)',
        r'therefore[,:\s]+(.+?)(?:\.|$)',
    ]
    for pattern in patterns:
        match = re.search(pattern, response, re.IGNORECASE | re.DOTALL)
        if match:
            return match.group(1).strip()

    sentences = [s.strip() for s in response.rstrip('.').split('.') if s.strip()]
    return sentences[-1] if sentences else response

def normalize(text):
    text = normalize_latex(text)
    return text.lower().strip().rstrip('.')

In [17]:
rouge_l_scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

def exact_match(prediction, reference):
    return normalize(extract_answer(prediction)) == normalize(reference)

def contains_match(prediction, reference):
    return normalize(reference) in normalize(prediction)

def compute_rouge_l(prediction, reference):
    return rouge_l_scorer.score(reference, prediction)['rougeL'].fmeasure

def compute_f1_token(prediction, reference):
    pred_tokens = normalize(extract_answer(prediction)).split()
    ref_tokens = normalize(reference).split()
    if not pred_tokens or not ref_tokens:
        return 0.0
    pred_counts = Counter(pred_tokens)
    ref_counts = Counter(ref_tokens)
    common = sum((pred_counts & ref_counts).values())
    if not common:
        return 0.0
    precision = common / len(pred_tokens)
    recall = common / len(ref_tokens)
    return 2 * precision * recall / (precision + recall)

In [18]:
has_ref = df[df['answer_type'] != 'no_answer'].copy()
has_ref['extracted_answer'] = has_ref['response'].apply(extract_answer)
has_ref['exact_match'] = has_ref.apply(lambda r: exact_match(r['response'], r['reference_answer']), axis=1)
has_ref['contains_match'] = has_ref.apply(lambda r: contains_match(r['response'], r['reference_answer']), axis=1)
has_ref['rouge_l'] = has_ref.apply(lambda r: compute_rouge_l(r['response'], r['reference_answer']), axis=1)
has_ref['f1_token'] = has_ref.apply(lambda r: compute_f1_token(r['response'], r['reference_answer']), axis=1)

has_ref['has_marker'] = has_ref['response'].str.contains(ANSWER_MARKER, regex=False)
print("ANSWER: marker hit rate by technique:")
print(has_ref.groupby('technique')['has_marker'].mean().round(4))
print()

has_ref.groupby('technique').agg(
    exact_match=('exact_match', 'mean'),
    contains_match=('contains_match', 'mean'),
    rouge_l=('rouge_l', 'mean'),
    f1_token=('f1_token', 'mean'),
    avg_response_len=('response_length', 'mean'),
    avg_gen_time=('generation_time', 'mean'),
).round(4)

ANSWER: marker hit rate by technique:
technique
dynamic_few_shot        0.0122
dynamic_few_shot_cot    0.1463
Name: has_marker, dtype: float64



,exact_match,contains_match,rouge_l,f1_token,avg_response_len,avg_gen_time
technique,,,,,,
dynamic_few_shot,0.0,0.0854,0.0464,0.0834,547.2317,36.6764
dynamic_few_shot_cot,0.0,0.0732,0.0468,0.1040,535.1951,62.2344


## Next step

Sanity metrics above are indicative only (no BERTScore here). Run `notebook_comparison.ipynb` once all three `results_<model>_dynamic.json` files are in Drive `NLP Project/Results/` — it computes the full metric set for fixed **and** dynamic runs with the same code path, plus paired significance tests, answer-type breakdowns, retrieval-quality analysis, and flip analysis.